Check the implementation of various models

In [126]:
from reload import reload_package
reload_package(namespace=globals())

Reloaded 3 modules, rebound 3 names.


In [2]:
import altair as alt
import numpy as np
import polars as pl
import torch

alt.renderers.enable("jupyter", offline=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data

In [73]:
from ts_uncertainty.plotting import plot_synthetic_sample
from ts_uncertainty.synthetic import SyntheticGenerator, SyntheticSample

In [108]:
context_length = 64
horizon_length = 12

sin_gen = SyntheticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    seasonal=True,
    covariate=True,
    periods=(24,),             # period of the sine
    n_periods=1,
    n_harmonics=1,             # 1 harmonic = pure sine, no extra shape
    level_range=(0.0, 0.0),    # centered at 0
    slope_range=(0.0, 0.0),    # no trend
    amplitude_range=(1.0, 1.0),
    noise=False,
    spikes=False,
    shift=False,
)
s = sin_gen.sample()
plot_synthetic_sample(s, title="Sin Sample")

# Statsforecasting

In [113]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

In [109]:
def sample_to_sf_input(s: SyntheticSample, uid: str = "0"):
    """
    returns:
      df:     long df (unique_id, ds, y, cov_*) over the history, length T
      X_df:   future covariates (unique_id, ds, cov_*), length H, or None
      target: (1, H) ground truth
    """
    x = np.asarray(s.context, dtype=np.float32)
    y = np.asarray(s.target, dtype=np.float32)
    T, H = len(x), len(y)

    df_cols = {"unique_id": [uid] * T, "ds": np.arange(T), "y": x}
    X_df = None

    if s.covariate is not None and np.size(s.covariate) > 0:
        c = np.asarray(s.covariate, dtype=np.float32)
        if c.ndim == 1:
            c = c[:, None]
        assert c.shape[0] >= T + H, "covariates must cover context + horizon"
        cov_names = [f"cov_{i}" for i in range(c.shape[1])]
        df_cols |= {n: c[:T, i] for i, n in enumerate(cov_names)}
        X_df = pl.DataFrame(
            {"unique_id": [uid] * H, "ds": np.arange(T, T + H)}
            | {n: c[T:T + H, i] for i, n in enumerate(cov_names)}
        )

    return pl.DataFrame(df_cols), X_df, torch.from_numpy(y).unsqueeze(0)

def samples_to_sf_inputs(samples):
    dfs, X_dfs, ys = zip(*(sample_to_sf_input(s, uid=str(i)) for i, s in enumerate(samples)))
    df = pl.concat(dfs)
    X_df = None if X_dfs[0] is None else pl.concat(X_dfs)
    return df, X_df, torch.stack(ys)    # (B, 1, H)

# model input and output shapes
x, c, y = samples_to_sf_inputs([s])

In [127]:
sf = StatsForecast(
    models=[AutoARIMA(season_length=1)],
    freq=1,
)
fc = sf.forecast(df=x, h=horizon_length, X_df=c, level=[90])
point_fc = fc["AutoARIMA"].to_numpy()
plot_synthetic_sample(s, title="Sin Sample", prediction=point_fc, quantiles=None, intervals=None, width=800)

# Chronos2

In [6]:
from chronos import Chronos2Pipeline, BaseChronosPipeline

In [ ]:
def sample_to_chronos2_input(s: SyntheticSample):
    """
    s.context:   (T,)               target history
    s.covariate: (T + H, V) or None covariates over history + forecast horizon
    s.target:    (H,)               ground truth over the forecast horizon
    returns:
      inp:    dict for Chronos2Pipeline.predict
      target: (1, H)                ground truth, shaped like one target variate
    """
    x = np.asarray(s.context, dtype=np.float32)
    y = np.asarray(s.target, dtype=np.float32)
    T, H = len(x), len(y)

    inp = {"target": x}

    if s.covariate is not None and np.size(s.covariate) > 0:
        c = np.asarray(s.covariate, dtype=np.float32)
        if c.ndim == 1:
            c = c[:, None]                      # single covariate given as (T+H,)
        assert c.shape[0] >= T + H, "covariates must cover context + horizon"
        V = c.shape[1]
        inp["past_covariates"] = {f"cov_{i}": c[:T, i] for i in range(V)}
        inp["future_covariates"] = {f"cov_{i}": c[T:T + H, i] for i in range(V)}

    return inp, torch.from_numpy(y).unsqueeze(0)

def sample_to_chronos2_inputs(samples):
    inputs, targets = zip(*(sample_to_chronos2_input(s) for s in samples))
    return list(inputs), torch.stack(targets)    # list of B dicts, (B, 1, H)

# model input and output shapes
x, y = sample_to_chronos2_inputs([s])

In [77]:
# loading model
chronos_pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device)
chronos_config = chronos_pipeline.model.chronos_config
print(chronos_config)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Chronos2ForecastingConfig(context_length=8192, output_patch_size=16, input_patch_size=16, input_patch_stride=16, quantiles=[0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99], use_reg_token=True, use_arcsinh=True, max_output_patches=64, time_encoding_scale=8192)


In [102]:
# predict
chronos_out = chronos_pipeline.predict(x, prediction_length=horizon_length, cross_learning=False)
print (chronos_out[0].shape)
plot_synthetic_sample(s, title="Sin Sample", prediction=chronos_out[0], quantiles=chronos_config.quantiles, intervals=[0.1,0.9], width=800)

torch.Size([1, 21, 12])


# TiRex-2

# Timestral